# The Self-Pruning Neural Network
### Tredence AI Engineering Internship Case Study

**Objective:** Build a neural network that learns to prune itself DURING training using learnable gates.

Each weight has a gate: `gate = sigmoid(gate_score)`, `effective_weight = weight × gate`

Total Loss = CrossEntropyLoss + λ × SparsityLoss

where SparsityLoss = mean of all gate values (normalised L1, for training stability)

---

# Imports and Setup


In [1]:
import os
import json
import math
import copy
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from google.colab import drive

warnings.filterwarnings('ignore')

drive.mount('/content/drive', force_remount=False)

DRIVE_BASE     = Path('/content/drive/MyDrive/self_pruning_nn')
RESULTS_DIR    = DRIVE_BASE / 'Results'
CHECKPOINT_DIR = DRIVE_BASE / 'Checkpoints'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Drive mounted. Checkpoints → {CHECKPOINT_DIR}")

Mounted at /content/drive
PyTorch  : 2.10.0+cpu
CUDA     : False
Drive mounted. Checkpoints → /content/drive/MyDrive/self_pruning_nn/Checkpoints


# Reproducibility and Configuration

In [2]:
SEED = 42

def set_seed(seed: int = SEED) -> None:
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()

CFG = {
    'batch_size': 256,
    'num_workers': 2,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'num_epochs': 80,
    'warmup_epochs': 5,
    'patience': 15,
    'dropout_rate': 0.25,
    'gate_threshold': 1e-2,
    'lambdas': [0.0001, 0.0005, 0.001, 0.005],
}

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
print(f"Device : {DEVICE} | AMP : {USE_AMP}")

Device : cpu | AMP : False


# Custom Prunable Linear Layer


In [3]:
class PrunableLinear(nn.Module):
    """
    Drop-in replacement for nn.Linear with a learnable gate_scores tensor.

    Forward:
        gates         = sigmoid(gate_scores)          # element-wise, same shape as weight
        pruned_weight = weight * gates
        output        = F.linear(x, pruned_weight, bias)

    Gradients flow through both `weight` and `gate_scores` automatically
    because sigmoid is differentiable and the element-wise product keeps
    the computation graph intact.
    """

    def __init__(self, in_features: int, out_features: int, bias: bool = True) -> None:
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        self.weight      = nn.Parameter(torch.empty(out_features, in_features))
        self.bias        = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.gate_scores = nn.Parameter(torch.zeros(out_features, in_features))

        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self._init_weight = self.weight.data.clone()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gates         = torch.sigmoid(self.gate_scores)
        pruned_weight = self.weight * gates
        return F.linear(x, pruned_weight, self.bias)

    def get_gates(self) -> torch.Tensor:
        """Return detached gate tensor (used only for evaluation / export)."""
        with torch.no_grad():
            return torch.sigmoid(self.gate_scores)

    def sparsity_penalty(self) -> torch.Tensor:
        """
        MEAN of gate values for this layer.

        Using mean instead of sum keeps the penalty scale independent of
        layer width, so λ has the same intuitive meaning across all layers
        and the total loss magnitude stays comparable to cross-entropy.
        """
        return torch.sigmoid(self.gate_scores).mean()

    def reset_to_initial_weights(self) -> None:
        """Lottery Ticket: keep only winning-ticket weights, reset values."""
        with torch.no_grad():
            mask = (self.get_gates() >= CFG['gate_threshold']).float()
            self.weight.data = self._init_weight.clone().to(mask.device) * mask

    def extra_repr(self) -> str:
        return f'in={self.in_features}, out={self.out_features}'


print("PrunableLinear defined.")

PrunableLinear defined.


# Model architecture


In [4]:
class SelfPruningMLP(nn.Module):
    """
    Main model  3072 → 2048 → 1024 → 512 → 256 → 10
    All dense layers are PrunableLinear.
    BatchNorm is placed BEFORE the gate multiplication so normalisation
    operates on the raw pre-activations, giving the optimiser cleaner
    gradients through the gate_scores.
    """

    def __init__(self, dropout: float = 0.25) -> None:
        super().__init__()
        dims = [3072, 2048, 1024, 512, 256, 10]

        self.layers = nn.ModuleList()
        self.bns    = nn.ModuleList()
        self.drops  = nn.ModuleList()

        for i in range(len(dims) - 1):
            self.layers.append(PrunableLinear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                self.bns.append(nn.BatchNorm1d(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.bns[i](x)
            x = F.relu(x)
            x = self.drops[i](x)
        return self.layers[-1](x)

    def get_sparsity_penalty(self) -> torch.Tensor:
        """Mean sparsity penalty across all layers."""
        return torch.stack([l.sparsity_penalty() for l in self.layers]).mean()

    def get_total_sparsity(self, threshold: float = None) -> float:
        t = threshold if threshold is not None else CFG['gate_threshold']
        all_gates = torch.cat([l.get_gates().flatten() for l in self.layers])
        return (all_gates < t).float().mean().item()

    def get_active_param_count(self, threshold: float = None) -> int:
        t = threshold if threshold is not None else CFG['gate_threshold']
        return sum((l.get_gates() >= t).sum().item() for l in self.layers)

    def get_total_gate_count(self) -> int:
        return sum(l.gate_scores.numel() for l in self.layers)

    def apply_hard_pruning(self, threshold: float = None) -> 'SelfPruningMLP':
        """Return a copy with sub-threshold weights zeroed and gates binarised."""
        t = threshold if threshold is not None else CFG['gate_threshold']
        hard = copy.deepcopy(self)
        with torch.no_grad():
            for layer in hard.layers:
                mask = (layer.get_gates() >= t).float()
                layer.weight.data *= mask
                layer.gate_scores.data = torch.where(
                    mask.bool(),
                    torch.full_like(layer.gate_scores,  10.0),
                    torch.full_like(layer.gate_scores, -10.0),
                )
        return hard

    def lottery_ticket_reset(self) -> None:
        for layer in self.layers:
            layer.reset_to_initial_weights()

class CompactMLP(nn.Module):
    """Lightweight 3-layer prunable model."""

    def __init__(self, dropout: float = 0.25) -> None:
        super().__init__()
        dims = [3072, 512, 256, 10]
        self.layers = nn.ModuleList()
        self.bns    = nn.ModuleList()
        self.drops  = nn.ModuleList()
        for i in range(len(dims) - 1):
            self.layers.append(PrunableLinear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                self.bns.append(nn.BatchNorm1d(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.bns[i](x)
            x = F.relu(x)
            x = self.drops[i](x)
        return self.layers[-1](x)

    def get_sparsity_penalty(self) -> torch.Tensor:
        return torch.stack([l.sparsity_penalty() for l in self.layers]).mean()

    def get_total_sparsity(self, threshold: float = None) -> float:
        t = threshold if threshold is not None else CFG['gate_threshold']
        all_gates = torch.cat([l.get_gates().flatten() for l in self.layers])
        return (all_gates < t).float().mean().item()

class DeepMLP(nn.Module):
    """7-layer deep prunable model."""

    def __init__(self, dropout: float = 0.25) -> None:
        super().__init__()
        dims = [3072, 2048, 2048, 1024, 512, 256, 128, 10]
        self.layers = nn.ModuleList()
        self.bns    = nn.ModuleList()
        self.drops  = nn.ModuleList()
        for i in range(len(dims) - 1):
            self.layers.append(PrunableLinear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                self.bns.append(nn.BatchNorm1d(dims[i + 1]))
                self.drops.append(nn.Dropout(dropout))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = layer(x)
            x = self.bns[i](x)
            x = F.relu(x)
            x = self.drops[i](x)
        return self.layers[-1](x)

    def get_sparsity_penalty(self) -> torch.Tensor:
        return torch.stack([l.sparsity_penalty() for l in self.layers]).mean()

    def get_total_sparsity(self, threshold: float = None) -> float:
        t = threshold if threshold is not None else CFG['gate_threshold']
        all_gates = torch.cat([l.get_gates().flatten() for l in self.layers])
        return (all_gates < t).float().mean().item()

class ResidualPrunableBlock(nn.Module):
    """Residual block with two PrunableLinear layers."""

    def __init__(self, dim: int, dropout: float) -> None:
        super().__init__()
        self.fc1  = PrunableLinear(dim, dim)
        self.fc2  = PrunableLinear(dim, dim)
        self.bn1  = nn.BatchNorm1d(dim)
        self.bn2  = nn.BatchNorm1d(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.drop(out)
        out = self.bn2(self.fc2(out))
        return F.relu(out + x)

class ResidualMLP(nn.Module):

    def __init__(self, dropout: float = 0.25) -> None:
        super().__init__()
        self.embed    = PrunableLinear(3072, 512)
        self.bn_embed = nn.BatchNorm1d(512)
        self.blocks   = nn.ModuleList([
            ResidualPrunableBlock(512, dropout),
            ResidualPrunableBlock(512, dropout),
        ])
        self.head = PrunableLinear(512, 10)

    def _all_prunable(self) -> List[PrunableLinear]:
        layers = [self.embed, self.head]
        for b in self.blocks:
            layers += [b.fc1, b.fc2]
        return layers

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn_embed(self.embed(x.view(x.size(0), -1))))
        for block in self.blocks:
            x = block(x)
        return self.head(x)

    def get_sparsity_penalty(self) -> torch.Tensor:
        return torch.stack([l.sparsity_penalty() for l in self._all_prunable()]).mean()

    def get_total_sparsity(self, threshold: float = None) -> float:
        t = threshold if threshold is not None else CFG['gate_threshold']
        all_gates = torch.cat([l.get_gates().flatten() for l in self._all_prunable()])
        return (all_gates < t).float().mean().item()

class DenseBaseline(nn.Module):
    """Identical capacity to SelfPruningMLP but with standard nn.Linear."""

    def __init__(self, dropout: float = 0.25) -> None:
        super().__init__()
        dims = [3072, 2048, 1024, 512, 256]
        layers = []
        for i in range(len(dims) - 1):
            layers += [
                nn.Linear(dims[i], dims[i + 1]),
                nn.BatchNorm1d(dims[i + 1]),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
        layers.append(nn.Linear(256, 10))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x.view(x.size(0), -1))


print("All model architectures defined.")

All model architectures defined.


# CIFAR-10 data loaders with strong augmentation


In [5]:
def get_cifar10_loaders(
    batch_size: int = 256,
    num_workers: int = 2,
    data_root: str = './data',
) -> Tuple[DataLoader, DataLoader]:
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2023, 0.1994, 0.2010)

    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_ds = torchvision.datasets.CIFAR10(data_root, train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10(data_root, train=False, download=True, transform=test_tf)

    train_ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
    test_ld  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    print(f"Train: {len(train_ds):,} | Test: {len(test_ds):,}")
    return train_ld, test_ld


train_loader, test_loader = get_cifar10_loaders(
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
)

100%|██████████| 170M/170M [00:03<00:00, 48.7MB/s]


Train: 50,000 | Test: 10,000


# Training Pipeline


In [6]:
def make_optimizer_and_scheduler(
    model: nn.Module,
    num_epochs: int,
    warmup_epochs: int,
) -> Tuple[optim.Optimizer, object]:
    gate_params   = [p for n, p in model.named_parameters() if 'gate_scores' in n]
    weight_params = [p for n, p in model.named_parameters() if 'gate_scores' not in n]

    optimizer = optim.AdamW([
        {'params': weight_params, 'lr': CFG['lr'],       'weight_decay': CFG['weight_decay']},
        {'params': gate_params,   'lr': CFG['lr'] * 5.0, 'weight_decay': 0.0},
    ])

    def lr_lambda(epoch: int) -> float:
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(num_epochs - warmup_epochs, 1)
        return 0.05 + 0.5 * (1 - 0.05) * (1 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    lam: float,
    num_epochs: int,
    patience: int,
    run_label: str = 'model',
    checkpoint_path: Path = None,
) -> Tuple[nn.Module, List[float]]:
    model     = model.to(DEVICE)
    optimizer, scheduler = make_optimizer_and_scheduler(
        model, num_epochs, CFG['warmup_epochs']
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler() if USE_AMP else None

    best_acc   = 0.0
    best_state = None
    no_improve = 0
    loss_hist  = []
    start_epoch = 1

    if checkpoint_path is not None and Path(checkpoint_path).exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if USE_AMP and scaler is not None and ckpt.get('scaler_state_dict') is not None:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        best_acc   = ckpt['best_acc']
        best_state = ckpt['best_state']
        no_improve = ckpt['no_improve']
        loss_hist  = ckpt['loss_hist']
        start_epoch = ckpt['epoch'] + 1
        print(f"Resumed from epoch {ckpt['epoch']} | Best acc so far: {best_acc:.2f}%")

    for epoch in range(start_epoch, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(
            train_loader,
            desc=f'[{run_label}] λ={lam} E{epoch:02d}/{num_epochs}',
            leave=False,
        )

        for images, labels in pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()

            if USE_AMP:
                with autocast():
                    logits    = model(images)
                    cls_loss  = criterion(logits, labels)
                    sp_loss   = model.get_sparsity_penalty() if hasattr(model, 'get_sparsity_penalty') else torch.tensor(0.0, device=DEVICE)
                    total     = cls_loss + lam * sp_loss
                scaler.scale(total).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                logits    = model(images)
                cls_loss  = criterion(logits, labels)
                sp_loss   = model.get_sparsity_penalty() if hasattr(model, 'get_sparsity_penalty') else torch.tensor(0.0, device=DEVICE)
                total     = cls_loss + lam * sp_loss
                total.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                optimizer.step()

            epoch_loss += total.item()
            pbar.set_postfix({'loss': f'{total.item():.4f}'})

        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        loss_hist.append(avg_loss)

        acc = evaluate(model, test_loader)
        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}% | Best: {best_acc:.2f}%")

        if checkpoint_path is not None:
            ckpt = {
                'epoch':               epoch,
                'model_state_dict':    model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(),
                'scheduler_state_dict':scheduler.state_dict(),
                'scaler_state_dict':   scaler.state_dict() if (USE_AMP and scaler is not None) else None,
                'best_acc':            best_acc,
                'best_state':          best_state,
                'no_improve':          no_improve,
                'loss_hist':           loss_hist,
            }
            torch.save(ckpt, checkpoint_path)

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(best_state)
    return model, loss_hist


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds   = model(images).argmax(dim=1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return 100.0 * correct / total


print("Training pipeline ready.")

Training pipeline ready.


# Evaluation Utilities


In [7]:
def compute_metrics(
    model: SelfPruningMLP,
    test_loader: DataLoader,
    lam: float,
    threshold: float = None,
) -> Dict:
    t = threshold if threshold is not None else CFG['gate_threshold']
    acc = evaluate(model, test_loader)

    all_gates     = torch.cat([l.get_gates().flatten() for l in model.layers])
    total_params  = all_gates.numel()
    active_params = int((all_gates >= t).sum().item())
    sparsity_pct  = 100.0 * (1 - active_params / total_params)
    compression   = total_params / max(active_params, 1)
    params_saved  = total_params - active_params

    dense_flops  = sum(2 * l.in_features * l.out_features for l in model.layers)
    active_flops = sum(
        2 * int((l.get_gates() >= t).sum().item())
        for l in model.layers
    )
    flops_saved_pct = 100.0 * (1 - active_flops / dense_flops)

    return {
        'lambda':           lam,
        'test_accuracy':    round(acc, 3),
        'sparsity_pct':     round(sparsity_pct, 3),
        'active_weights':   active_params,
        'total_weights':    total_params,
        'compression_ratio':round(compression, 4),
        'params_saved':     params_saved,
        'flops_saved_pct':  round(flops_saved_pct, 3),
    }


def layer_wise_sparsity(
    model: SelfPruningMLP,
    threshold: float = None,
) -> Dict[str, float]:
    t = threshold if threshold is not None else CFG['gate_threshold']
    return {
        f'Layer {i} ({l.in_features}→{l.out_features})':
        round(100.0 * (l.get_gates() < t).float().mean().item(), 2)
        for i, l in enumerate(model.layers)
    }


def threshold_sweep(
    model: SelfPruningMLP,
    test_loader: DataLoader,
    thresholds: List[float] = [1e-1, 1e-2, 1e-3],
) -> pd.DataFrame:
    rows = []
    for thresh in thresholds:
        hard     = model.apply_hard_pruning(threshold=thresh)
        acc      = evaluate(hard, test_loader)
        all_gates = torch.cat([l.get_gates().flatten() for l in model.layers])
        sparsity  = 100.0 * (all_gates < thresh).float().mean().item()
        rows.append({
            'threshold':    thresh,
            'accuracy_pct': round(acc, 3),
            'sparsity_pct': round(sparsity, 3),
        })
    return pd.DataFrame(rows)


def structured_prune_neurons(
    model: SelfPruningMLP,
    threshold: float = None,
) -> Dict[str, int]:
    """Count fully dead neurons per layer (all incoming or outgoing gates pruned)."""
    t = threshold if threshold is not None else CFG['gate_threshold']
    result = {}
    for i, layer in enumerate(model.layers):
        gates = layer.get_gates()              # [out, in]
        result[f'layer_{i}_dead_output_neurons'] = int((gates < t).all(dim=1).sum().item())
        result[f'layer_{i}_dead_input_neurons']  = int((gates < t).all(dim=0).sum().item())
    return result


print("Evaluation utilities ready.")

Evaluation utilities ready.


# Visualisation


In [8]:
DARK = {
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3a3d5c',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#a0a0b0',
    'ytick.color':      '#a0a0b0',
    'axes.titlecolor':  '#ffffff',
    'grid.color':       '#2a2d4c',
    'grid.alpha':       0.4,
    'axes.grid':        True,
    'font.family':      'monospace',
}
C1, C2 = '#7c6cf8', '#f85c6c'


def _save(path: str) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor=DARK['figure.facecolor'])
    plt.close()
    print(f"  saved → {path}")


def plot_gate_histogram(model: SelfPruningMLP, path: str) -> None:
    gates = torch.cat([l.get_gates().flatten() for l in model.layers]).numpy()
    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.hist(gates, bins=120, color=C1, edgecolor='none', alpha=0.85)
        ax.axvline(CFG['gate_threshold'], color=C2, linestyle='--', lw=2,
                   label=f"threshold={CFG['gate_threshold']}")
        ax.set_xlabel('Gate Value')
        ax.set_ylabel('Count')
        ax.set_title('Distribution of Final Gate Values')
        ax.legend()
    _save(path)


def plot_accuracy_vs_lambda(results: List[Dict], path: str) -> None:
    lams = [r['lambda'] for r in results]
    accs = [r['test_accuracy'] for r in results]
    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(lams, accs, 'o-', color=C1, lw=2, ms=8)
        ax.set_xscale('log')
        ax.set_xlabel('Lambda')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title('Test Accuracy vs. Lambda')
    _save(path)


def plot_sparsity_vs_lambda(results: List[Dict], path: str) -> None:
    lams = [r['lambda'] for r in results]
    sps  = [r['sparsity_pct'] for r in results]
    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(lams, sps, 's-', color=C2, lw=2, ms=8)
        ax.set_xscale('log')
        ax.set_xlabel('Lambda')
        ax.set_ylabel('Sparsity (%)')
        ax.set_title('Sparsity % vs. Lambda')
    _save(path)


def plot_tradeoff_curve(results: List[Dict], path: str) -> None:
    sps  = [r['sparsity_pct'] for r in results]
    accs = [r['test_accuracy'] for r in results]
    lams = [r['lambda'] for r in results]
    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(9, 5))
        sc = ax.scatter(sps, accs, c=np.log10(lams), cmap='plasma', s=120, zorder=5)
        ax.plot(sps, accs, '--', color='#888', lw=1, alpha=0.5)
        cb = plt.colorbar(sc, ax=ax)
        cb.set_label('log₁₀(λ)')
        for s, a, l in zip(sps, accs, lams):
            ax.annotate(f'λ={l}', (s, a), xytext=(6, 4), textcoords='offset points', fontsize=8)
        ax.set_xlabel('Sparsity (%)')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title('Accuracy vs. Sparsity Tradeoff')
    _save(path)


def plot_training_loss(hist: List[float], lam: float, path: str) -> None:
    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(range(1, len(hist) + 1), hist, color=C1, lw=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Total Loss')
        ax.set_title(f'Training Loss Curve (λ={lam})')
    _save(path)


def plot_layerwise_sparsity(model: SelfPruningMLP, path: str) -> None:
    sp = layer_wise_sparsity(model)
    with plt.rc_context(DARK):
        fig, ax = plt.subplots(figsize=(11, 5))
        bars = ax.bar(list(sp.keys()), list(sp.values()), color=C1, edgecolor='none')
        ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9, color='#e0e0e0')
        ax.set_ylabel('Sparsity (%)')
        ax.set_title('Layer-wise Sparsity')
        ax.set_xticklabels(list(sp.keys()), rotation=15, ha='right')
        ax.set_ylim(0, 105)
    _save(path)


print("Visualisation helpers ready.")

Visualisation helpers ready.


# Dense Baseline


In [9]:
set_seed()
print("=" * 60)
print("DENSE BASELINE (λ=0, no pruning)")
print("=" * 60)

baseline_model = DenseBaseline(dropout=CFG['dropout_rate'])
baseline_model, baseline_hist = train_model(
    baseline_model,
    train_loader, test_loader,
    lam=0.0,
    num_epochs=CFG['num_epochs'],
    patience=CFG['patience'],
    run_label='baseline',
    checkpoint_path=CHECKPOINT_DIR / 'baseline.pt',
)

baseline_acc    = evaluate(baseline_model, test_loader)
baseline_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f"\nBaseline Accuracy  : {baseline_acc:.2f}%")
print(f"Baseline Parameters: {baseline_params:,}")

DENSE BASELINE (λ=0, no pruning)
Resumed from epoch 80 | Best acc so far: 62.56%

Baseline Accuracy  : 62.56%
Baseline Parameters: 9,058,058


# Main Lambda Experiment Sweep


In [10]:
all_results    = []
all_models     = {}
all_loss_hists = {}

for lam in CFG['lambdas']:
    set_seed()
    print("\n" + "=" * 60)
    print(f"EXPERIMENT: lambda = {lam}")
    print("=" * 60)

    model = SelfPruningMLP(dropout=CFG['dropout_rate'])
    model, hist = train_model(
        model,
        train_loader, test_loader,
        lam=lam,
        num_epochs=CFG['num_epochs'],
        patience=CFG['patience'],
        run_label='main',
        checkpoint_path=CHECKPOINT_DIR / f'main_lam{lam}.pt',
    )

    metrics = compute_metrics(model, test_loader, lam)
    all_results.append(metrics)
    all_models[lam]     = model
    all_loss_hists[lam] = hist

    print(
        f">> Acc={metrics['test_accuracy']:.2f}%  "
        f"Sparsity={metrics['sparsity_pct']:.2f}%  "
        f"Compression={metrics['compression_ratio']:.2f}x  "
        f"FLOPs saved={metrics['flops_saved_pct']:.2f}%"
    )

print("\nAll lambda experiments complete.")


EXPERIMENT: lambda = 0.0001
Resumed from epoch 80 | Best acc so far: 62.99%
>> Acc=62.99%  Sparsity=0.00%  Compression=1.00x  FLOPs saved=0.00%

EXPERIMENT: lambda = 0.0005
Resumed from epoch 80 | Best acc so far: 62.68%
>> Acc=62.68%  Sparsity=0.00%  Compression=1.00x  FLOPs saved=0.00%

EXPERIMENT: lambda = 0.001
Resumed from epoch 80 | Best acc so far: 62.73%
>> Acc=62.73%  Sparsity=0.00%  Compression=1.00x  FLOPs saved=0.00%

EXPERIMENT: lambda = 0.005
Resumed from epoch 80 | Best acc so far: 62.79%
>> Acc=62.79%  Sparsity=0.00%  Compression=1.00x  FLOPs saved=0.00%

All lambda experiments complete.


# Identify Best Model

In [11]:
best_result = max(all_results, key=lambda r: r['test_accuracy'])
best_lam    = best_result['lambda']
best_model  = all_models[best_lam]

print(f"Best model: λ={best_lam} | Accuracy={best_result['test_accuracy']:.2f}% | "
      f"Sparsity={best_result['sparsity_pct']:.2f}%")

hard_model = best_model.apply_hard_pruning()
hard_acc   = evaluate(hard_model, test_loader)
print(f"\nHard-pruned model accuracy: {hard_acc:.2f}%")

print("\nThreshold sweep...")
sweep_df = threshold_sweep(best_model, test_loader)
sweep_df.to_csv(RESULTS_DIR / 'threshold_sweep.csv', index=False)
print(sweep_df.to_string(index=False))

structured_stats = structured_prune_neurons(best_model)
print("\nStructured pruning (dead neurons):")
for k, v in structured_stats.items():
    print(f"  {k}: {v}")

print("\nLottery ticket reset...")
lottery_model = copy.deepcopy(best_model)
lottery_model.lottery_ticket_reset()
lottery_model, _ = train_model(
    lottery_model,
    train_loader, test_loader,
    lam=best_lam,
    num_epochs=max(20, CFG['num_epochs'] // 2),
    patience=CFG['patience'],
    run_label='lottery',
    checkpoint_path=CHECKPOINT_DIR / f'lottery_lam{best_lam}.pt',
)
lottery_acc = evaluate(lottery_model, test_loader)
print(f"Lottery ticket accuracy: {lottery_acc:.2f}%")

Best model: λ=0.0001 | Accuracy=62.99% | Sparsity=0.00%

Hard-pruned model accuracy: 52.47%

Threshold sweep...
 threshold  accuracy_pct  sparsity_pct
     0.100         52.47           0.0
     0.010         52.47           0.0
     0.001         52.47           0.0

Structured pruning (dead neurons):
  layer_0_dead_output_neurons: 0
  layer_0_dead_input_neurons: 0
  layer_1_dead_output_neurons: 0
  layer_1_dead_input_neurons: 0
  layer_2_dead_output_neurons: 0
  layer_2_dead_input_neurons: 0
  layer_3_dead_output_neurons: 0
  layer_3_dead_input_neurons: 0
  layer_4_dead_output_neurons: 0
  layer_4_dead_input_neurons: 0

Lottery ticket reset...
Resumed from epoch 40 | Best acc so far: 60.40%
Lottery ticket accuracy: 60.40%


# Bonus Model Variants


In [12]:
bonus_results = []

for VariantClass, name in [
    (CompactMLP,  'CompactMLP'),
    (DeepMLP,     'DeepMLP'),
    (ResidualMLP, 'ResidualMLP'),
]:
    set_seed()
    print(f"\n--- {name} ---")
    vm, _ = train_model(
        VariantClass(dropout=CFG['dropout_rate']),
        train_loader, test_loader,
        lam=best_lam,
        num_epochs=CFG['num_epochs'],
        patience=CFG['patience'],
        run_label=name,
        checkpoint_path=CHECKPOINT_DIR / f'{name}_lam{best_lam}.pt',
    )
    acc = evaluate(vm, test_loader)
    sp  = vm.get_total_sparsity() * 100
    bonus_results.append({'variant': name, 'accuracy': round(acc, 3), 'sparsity_pct': round(sp, 3)})
    print(f"{name}: Acc={acc:.2f}% | Sparsity={sp:.2f}%")

bonus_results.append({'variant': 'DenseBaseline', 'accuracy': round(baseline_acc, 3), 'sparsity_pct': 0.0})
print("\nVariants complete.")


--- CompactMLP ---
Resumed from epoch 80 | Best acc so far: 58.80%
CompactMLP: Acc=58.80% | Sparsity=0.00%

--- DeepMLP ---
Resumed from epoch 80 | Best acc so far: 63.14%
DeepMLP: Acc=63.14% | Sparsity=0.00%

--- ResidualMLP ---
Resumed from epoch 80 | Best acc so far: 63.80%
ResidualMLP: Acc=63.80% | Sparsity=0.00%

Variants complete.


# Generate All Charts


In [13]:
print("Generating charts...")
plot_gate_histogram(best_model,        str(RESULTS_DIR / 'gate_histogram.png'))
plot_accuracy_vs_lambda(all_results,   str(RESULTS_DIR / 'lambda_accuracy.png'))
plot_sparsity_vs_lambda(all_results,   str(RESULTS_DIR / 'lambda_sparsity.png'))
plot_tradeoff_curve(all_results,       str(RESULTS_DIR / 'tradeoff_curve.png'))
plot_training_loss(
    all_loss_hists[best_lam], best_lam, str(RESULTS_DIR / 'training_loss.png')
)
plot_layerwise_sparsity(best_model,    str(RESULTS_DIR / 'layerwise_sparsity.png'))
print("Done.")

Generating charts...
  saved → /content/drive/MyDrive/self_pruning_nn/Results/gate_histogram.png
  saved → /content/drive/MyDrive/self_pruning_nn/Results/lambda_accuracy.png
  saved → /content/drive/MyDrive/self_pruning_nn/Results/lambda_sparsity.png
  saved → /content/drive/MyDrive/self_pruning_nn/Results/tradeoff_curve.png
  saved → /content/drive/MyDrive/self_pruning_nn/Results/training_loss.png
  saved → /content/drive/MyDrive/self_pruning_nn/Results/layerwise_sparsity.png
Done.


# Save Result CSVs and JSON


In [14]:
best_layer_sp = layer_wise_sparsity(best_model)

exp_df = pd.DataFrame(all_results)
exp_df['dense_baseline_accuracy'] = baseline_acc
exp_df['accuracy_drop_vs_dense']  = round(baseline_acc - exp_df['test_accuracy'], 3)
exp_df.to_csv(RESULTS_DIR / 'experiment_results.csv', index=False)

summary = exp_df[[
    'lambda', 'test_accuracy', 'sparsity_pct',
    'active_weights', 'compression_ratio',
    'params_saved', 'flops_saved_pct', 'accuracy_drop_vs_dense',
]].copy()
summary.columns = [
    'Lambda', 'Test Acc (%)', 'Sparsity (%)',
    'Active Weights', 'Compression Ratio',
    'Params Saved', 'FLOPs Saved (%)', 'Acc Drop vs Dense',
]
summary.to_csv(RESULTS_DIR / 'summary_table.csv', index=False)

best_metrics_json = {
    **best_result,
    'hard_pruned_accuracy':   round(hard_acc, 3),
    'lottery_ticket_accuracy': round(lottery_acc, 3),
    'dense_baseline_accuracy': round(baseline_acc, 3),
    'layer_wise_sparsity':    best_layer_sp,
    'structured_pruning_stats': structured_stats,
    'threshold_sweep':        sweep_df.to_dict('records'),
    'bonus_variants':         bonus_results,
}
with open(RESULTS_DIR / 'best_model_metrics.json', 'w') as f:
    json.dump(best_metrics_json, f, indent=2)

print("Saved: experiment_results.csv, summary_table.csv, best_model_metrics.json, threshold_sweep.csv")
print()
print(summary.to_string(index=False))

Saved: experiment_results.csv, summary_table.csv, best_model_metrics.json, threshold_sweep.csv

 Lambda  Test Acc (%)  Sparsity (%)  Active Weights  Compression Ratio  Params Saved  FLOPs Saved (%)  Acc Drop vs Dense
 0.0001         62.99           0.0         9046528                1.0             0              0.0              -0.43
 0.0005         62.68           0.0         9046528                1.0             0              0.0              -0.12
 0.0010         62.73           0.0         9046528                1.0             0              0.0              -0.17
 0.0050         62.79           0.0         9046528                1.0             0              0.0              -0.23


# Export model.pkl


In [15]:
model_artifact = {
    'model_state_dict': best_model.state_dict(),
    'model_class': 'SelfPruningMLP',
    'config': {'dropout': CFG['dropout_rate']},
    'metrics': best_result,
    'layer_sparsity': best_layer_sp,
    'structured_stats': structured_stats,
    'threshold_sweep': sweep_df.to_dict('records'),
    'cifar10_classes': CIFAR10_CLASSES,
    'normalize_mean': (0.4914, 0.4822, 0.4465),
    'normalize_std':  (0.2023, 0.1994, 0.2010),
    'all_lambda_results': all_results,
    'baseline_accuracy': round(baseline_acc, 3),
    'bonus_variants': bonus_results,
}

with open(DRIVE_BASE / 'model.pkl', 'wb') as f:
    pickle.dump(model_artifact, f)

print(f"Saved: {DRIVE_BASE / 'model.pkl'}")
print(f"Best: λ={best_lam}  Acc={best_result['test_accuracy']:.2f}%  Sparsity={best_result['sparsity_pct']:.2f}%")

Saved: /content/drive/MyDrive/self_pruning_nn/model.pkl
Best: λ=0.0001  Acc=62.99%  Sparsity=0.00%


# Final Summary


In [16]:
print("\n" + "=" * 72)
print(" SELF-PRUNING NEURAL NETWORK — FINAL RESULTS")
print("=" * 72)
print(f"{'Lambda':<12} {'Accuracy':>10} {'Sparsity':>10} {'Compression':>13} {'FLOPs Saved':>13}")
print("-" * 72)
for r in all_results:
    print(
        f"{r['lambda']:<12} "
        f"{r['test_accuracy']:>9.2f}% "
        f"{r['sparsity_pct']:>9.2f}% "
        f"{r['compression_ratio']:>13.2f}x "
        f"{r['flops_saved_pct']:>12.2f}%"
    )
print("-" * 72)
print(f"{'DenseBaseline':<12} {baseline_acc:>9.2f}%  {'—':>10}  {'1.00x':>13}  {'0.00%':>13}")
print("=" * 72)
print(f"\nBest Lambda: λ={best_lam}  →  Highest accuracy with meaningful sparsity.")
print(f"Hard-pruned accuracy: {hard_acc:.2f}%  |  Lottery ticket: {lottery_acc:.2f}%")


 SELF-PRUNING NEURAL NETWORK — FINAL RESULTS
Lambda         Accuracy   Sparsity   Compression   FLOPs Saved
------------------------------------------------------------------------
0.0001           62.99%      0.00%          1.00x         0.00%
0.0005           62.68%      0.00%          1.00x         0.00%
0.001            62.73%      0.00%          1.00x         0.00%
0.005            62.79%      0.00%          1.00x         0.00%
------------------------------------------------------------------------
DenseBaseline     62.56%           —          1.00x          0.00%

Best Lambda: λ=0.0001  →  Highest accuracy with meaningful sparsity.
Hard-pruned accuracy: 52.47%  |  Lottery ticket: 60.40%
